# U07 內幕（二）：索引與效能 ＋ ★示範：索引效能實驗

**資料庫管理**・統計系三年級・10/22　<a href="https://colab.research.google.com/github/chang-ye-tu/db/blob/master/notebooks/unit07.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

課程首頁：[github.com/chang-ye-tu/db](https://github.com/chang-ye-tu/db)・大綱：[syllabus.md](https://github.com/chang-ye-tu/db/blob/master/syllabus.md)・專題：[projects.md](https://github.com/chang-ye-tu/db/blob/master/projects.md)

百萬列裡 0.01 毫秒找到一筆——**B+ tree** 是怎麼辦到的；然後把「快多少」做成一個**正經的統計實驗**（你的專題共同要求 6 就是它的縮小版）。

> **投影片式 notebook 使用法**：上課跟著往下走，程式格按 `Shift+Enter` 執行；左側「目錄」可跳節。回家可以重跑、改參數做實驗——**講義是可以跑的**。
>
> 開始前建議：檔案 → 在雲端硬碟中儲存副本，改動才會留下來。

## 0. 本單元地圖（135 分鐘）

| 節 | 分鐘 | 內容 | 與專題的關係 |
|---|---|---|---|
| 第 1 節 | 50 | 百萬列的痛・索引是什麼・**自製排序索引（30 行）**・**B+ tree 現場實作與逐步分裂**・樹高公式・hash 的能與不能 | 懂了才調得動 |
| 第 2 節 | 50 | `CREATE INDEX`＋`EXPLAIN QUERY PLAN` 讀法・複合索引與最左前綴・覆蓋索引・運算式殺索引・selectivity・寫入代價・**★索引效能實驗研究（log-log）** | 共同要求 6 當堂學會 |
| 課堂實作 | 35 | 索引偵探（先預測再驗證）＋**對你自己的專題資料做前後對照** | 共同要求 6 完成 |

# 第 1 節：從「痛」到「B+ tree」

## 1.1 先痛一次：一百萬列裡找一個人

In [ ]:
#@title 📦 合成百萬列顧客表（固定 seed；約 4 秒）
import sqlite3, os, time
import numpy as np, pandas as pd

rng = np.random.default_rng(42)
N = 1_000_000
df = pd.DataFrame({
    "cid":    np.arange(1, N + 1),
    "city":   rng.choice(["台中", "台北", "高雄", "台南", "新竹", "桃園"], N,
                         p=[.3, .25, .15, .12, .1, .08]),
    "age":    rng.integers(18, 70, N),
    "amount": np.round(rng.lognormal(6, 0.8, N)).astype(int),
    "d":      (np.datetime64("2024-01-01") + rng.integers(0, 900, N)).astype(str),
})
if os.path.exists("million.db"):
    os.remove("million.db")
con = sqlite3.connect("million.db")
t = time.time()
df.to_sql("cust", con, index=False)
print(f"cust {N:,} 列 → million.db（{time.time()-t:.1f}s、{os.path.getsize('million.db')/1e6:.0f} MB）")

In [ ]:
# 沒有索引的點查：整表掃描（full scan）——資料庫「別無選擇」
def bench(sql, params=(), reps=5):
    t = time.time()
    for _ in range(reps):
        con.execute(sql, params).fetchall()
    return (time.time() - t) / reps * 1000            # ms／次

t_scan = bench("SELECT * FROM cust WHERE cid = ?", (777_777,))
print(f"WHERE cid = 777777 → {t_scan:.1f} ms／次")
print("查詢計畫自白：")
for r in con.execute("EXPLAIN QUERY PLAN SELECT * FROM cust WHERE cid = ?", (777_777,)):
    print("  ", r[3])                                  # SCAN cust ＝ 一百萬列全部翻過
print(f"\n→ 一次 {t_scan:.0f} ms 看似無感？你的 App 每個畫面查 10 次、54 個學生同時操作——馬上跪。")

## 1.2 索引是什麼：一本「按值排序的目錄」

| | 沒有索引 | 有索引 |
|---|---|---|
| 類比 | 從第 1 頁翻到第 N 頁找關鍵字 | 翻書後的**索引頁**：關鍵字→頁碼 |
| 結構 | 資料按寫入順序躺著 | **另外維護**一份「值 → 住址」的排序結構 |
| 代價 | 讀慢 | 寫入要順手更新目錄（2.6 量給你看） |

三個馬上要會的分類詞：

- **clustered（主樹）**：表本身就按主鍵排——SQLite 的表就是一棵按 `rowid` 排的 B-tree（U06 看過它躺在 page 裡）。
- **secondary（次級索引）**：`CREATE INDEX` 建的那些——另一棵樹，存「值 → rowid」，找到後**再回主樹撈整列**。
- dense／sparse：每筆都登記 vs 每頁登記一筆（教科書詞彙；SQLite 的都是 dense）。

**索引的靈魂只有一句：排好序，就能二分搜。** 先用 30 行 Python 把靈魂做出來：

In [ ]:
# 自製排序索引：sorted list + bisect ——「值 → 位置」的目錄
import bisect

class SortedIndex:
    def __init__(self, values):
        self.pairs = sorted((v, i) for i, v in enumerate(values))   # (值, rowid) 按值排序
        self.keys = [p[0] for p in self.pairs]

    def find(self, v):                                # 點查：二分搜
        i = bisect.bisect_left(self.keys, v)
        out = []
        while i < len(self.keys) and self.keys[i] == v:
            out.append(self.pairs[i][1]); i += 1
        return out

    def between(self, lo, hi):                        # 範圍查：找到起點後「順著走」
        i = bisect.bisect_left(self.keys, lo)
        j = bisect.bisect_right(self.keys, hi)
        return [self.pairs[k][1] for k in range(i, j)]

amounts = df["amount"].to_numpy()
vals = amounts.tolist()

t = time.time()
linear_hits = [i for i, v in enumerate(vals) if v == 12345]    # 沒目錄：全掃
linear_ms = (time.time() - t) * 1000

idx = SortedIndex(vals)                                        # 建目錄（一次性成本）
t = time.time()
for _ in range(1000):
    idx.find(12345)
bisect_ms = (time.time() - t)                                  # 1000 次共花的 ms

print(f"線性全掃 1 次　：{linear_ms:8.1f} ms（找到 {len(linear_hits)} 筆）")
print(f"二分搜 1000 次 ：{bisect_ms*1:8.1f} ms → 每次 {bisect_ms/1000*1000:.1f} µs")
print(f"範圍查 12000–12100：{len(idx.between(12000, 12100))} 筆（起點二分、之後順走——排序的紅利）")
assert sorted(idx.find(12345)) == linear_hits
print("✅ 跟全掃答案一致——索引不是魔法，是「排序＋二分」")

In [ ]:
# sorted list 的罩門：查得快，「插」得痛——每插一筆要挪動半條陣列
arr = sorted(rng.integers(0, 10**9, 500_000).tolist())
t = time.time()
for v in rng.integers(0, 10**9, 2000).tolist():
    bisect.insort(arr, v)                       # O(N) 搬家
ins_us = (time.time() - t) / 2000 * 1e6
t = time.time()
for v in rng.integers(0, 10**9, 2000).tolist():
    bisect.bisect_left(arr, v)                  # O(log N) 查
find_us = (time.time() - t) / 2000 * 1e6
print(f"50 萬筆的 sorted list：查一次 {find_us:.1f} µs，插一筆 {ins_us:.0f} µs（差 {ins_us/max(find_us,0.01):.0f} 倍）")
print("→ 查詢我們要、插入的痛不要——把陣列「切成小頁、疊成樹」就是解法：B+ tree。")

## 1.3 那為什麼不直接用 sorted list？——因為磁碟與增刪

sorted list 的兩個死穴：
1. **插入要挪動半條陣列**（O(N) 搬家）；
2. 它是一整條——磁碟世界裡什麼都要**切成 4KB 頁**（U06）。

**B+ tree ＝ 把排序陣列「切頁、疊層」的版本**：
- 資料全在**葉節點**（一片葉＝一頁），葉子之間串成鏈（範圍掃描直接順走）；
- 上層節點只放「路標」（分隔鍵）；
- 節點塞滿就**分裂**、把中間的路標往上推——樹**從根部長高**，永遠平衡。

口說無憑，現場做一棵（節點容量故意縮成 4，分裂才看得見）：

In [ ]:
ORDER = 4                       # 每個節點最多 4 個 key（真品是幾百——所以樹超矮）

class Node:
    __slots__ = ("leaf", "keys", "kids", "next")
    def __init__(self, leaf=True):
        self.leaf, self.keys, self.kids, self.next = leaf, [], [], None

class BPlusTree:
    def __init__(self):
        self.root = Node(leaf=True)

    def _leaf_for(self, k, want_trail=False):
        node, trail = self.root, []
        while not node.leaf:
            trail.append(node)
            node = node.kids[bisect.bisect_right(node.keys, k)]
        return (node, trail) if want_trail else node

    def insert(self, k):
        leaf, trail = self._leaf_for(k, want_trail=True)
        bisect.insort(leaf.keys, k)
        cur = leaf
        while len(cur.keys) > ORDER:                       # 爆了 → 分裂
            mid = len(cur.keys) // 2
            right = Node(leaf=cur.leaf)
            if cur.leaf:
                right.keys, cur.keys = cur.keys[mid:], cur.keys[:mid]
                up = right.keys[0]                         # 葉分裂：右邊第一個 key「複製」上推
                right.next, cur.next = cur.next, right
            else:
                up = cur.keys[mid]                         # 內部分裂：中間 key「搬」上去
                right.keys, cur.keys = cur.keys[mid+1:], cur.keys[:mid]
                right.kids, cur.kids = cur.kids[mid+1:], cur.kids[:mid+1]
            if trail:
                p = trail.pop()
                i = bisect.bisect_right(p.keys, up)
                p.keys.insert(i, up); p.kids.insert(i + 1, right)
                cur = p                                    # 爸爸可能也爆 → 繼續往上檢查
            else:
                new_root = Node(leaf=False)                # 沒有爸爸 → 長出新根（樹長高的唯一方式）
                new_root.keys, new_root.kids = [up], [cur, right]
                self.root = new_root
                break

    def search(self, k):
        return k in self._leaf_for(k).keys

    def show(self):
        level, depth = [self.root], 0
        while level:
            print(f"   L{depth}:", "  ".join("[" + " ".join(map(str, n.keys)) + "]" for n in level))
            if level[0].leaf:
                break
            level = [kid for n in level for kid in n.kids]
            depth += 1

print("B+ tree 就緒（insert／search／show；葉節點自帶串鏈）")

In [ ]:
# 逐步分裂直播：依序插入 1..10，盯著它「爆 → 裂 → 長高」
tree = BPlusTree()
for k in range(1, 11):
    tree.insert(k)
    if k in (4, 5, 8, 10):
        print(f"── 插入 {k} 之後 ──")
        tree.show()
        print()
print("看點：插入 5 時第一片葉爆掉 → 裂成兩片、長出新根；之後每次爆裂只把一個路標往上推。")

In [ ]:
# 亂序插入 300 個 key：照樣平衡；範圍掃描沿葉鏈直走
import random
random.seed(7)
tree = BPlusTree()
keys = random.sample(range(1000), 300)
for k in keys:
    tree.insert(k)

# 驗證：全部找得到、不存在的找不到
assert all(tree.search(k) for k in keys)
assert not any(tree.search(k) for k in set(range(1000)) - set(keys))

# 樹高（沿最左邊走到葉）
h, node = 0, tree.root
while not node.leaf:
    h += 1; node = node.kids[0]
print(f"300 個亂序 key → 樹高 {h}（根到葉 {h+1} 層）——平衡不是維護出來的，是「往上長」天生的")

# 範圍掃描：二分找到起點葉，然後沿 next 鏈直走（B+ tree 的招牌）
leaf0 = tree._leaf_for(100)
hits = []
while leaf0:
    hits += [k for k in leaf0.keys if 100 <= k <= 200]
    if leaf0.keys and leaf0.keys[-1] > 200:
        break
    leaf0 = leaf0.next
assert sorted(hits) == sorted(k for k in keys if 100 <= k <= 200)
print(f"BETWEEN 100 AND 200 → 沿葉鏈撈到 {len(hits)} 個 ✅（這就是範圍查詢快的原因）")

In [ ]:
# 用自己的樹驗證 fanout 定律：同樣 2,000 個 key，節點容量越大 → 樹越矮
def build_and_height(order, n=2000):
    global ORDER
    ORDER, old = order, ORDER
    t2 = BPlusTree()
    for k in random.Random(1).sample(range(10**6), n):
        t2.insert(k)
    h, node = 0, t2.root
    while not node.leaf:
        h += 1; node = node.kids[0]
    ORDER = old
    return h + 1                                   # 含葉那層

for f in (4, 8, 16, 64):
    print(f"節點容量 {f:3d} → 樹 {build_and_height(f)} 層")
print("→ 容量（fanout）翻倍，層數就往下掉——真實 4KB 頁 fanout 幾百，所以百萬列 3 層就到底。")

## 1.4 樹高公式：為什麼一百萬列只要 3 次 I/O

一個節點（＝一頁 4KB）能塞的路標數叫 **fanout（f）**。樹高 ≈ ⌈log_f N⌉：

| N | f=4（教學玩具） | f=250（4KB 頁的真實量級） |
|---|---|---|
| 1 千 | 5 | **2** |
| 100 萬 | 10 | **3** |
| 10 億 | 15 | **4** |

- 每往下走一層＝讀一頁；**十億筆也只要 4 頁**——而且上層那幾頁永遠熱在 buffer pool 裡（U06），實際常只碰 1 次磁碟。
- 這也解釋 U06 的伏筆「欄位別亂胖」：**key 越小 → fanout 越大 → 樹越矮**。

In [ ]:
# 隨堂練習 A0 工作區：算你自己的樹高——把 N 換成你專題的資料量，驗證「幾層到底」
import math
f = 250
for N_rows in (10_000, 100_000, 25_000_000):     # ← 把第一個換成你的
    print(f"N = {N_rows:>12,} → 樹高 ≈ {math.ceil(math.log(N_rows, f))} 層")
# 手感：f=250 時，每多一層就多吃 250 倍資料——所以「加一層」是很罕見的大事

## 1.5 hash index：更快的點查、換不到的範圍

| | B+ tree | hash |
|---|---|---|
| 點查 `=` | O(log_f N)（≈3 頁） | **O(1)** |
| 範圍 `BETWEEN`／前綴 `LIKE 'ab%'` | ✅ 葉鏈直走 | ❌ 打散了，只能全掃 |
| `ORDER BY` 白吃 | ✅ 本來就排好 | ❌ |

SQLite 的持久索引**只有 B-tree**（點查已經夠快、又保住範圍與排序）；hash 的主場在記憶體——
Python 的 dict、以及 U08 join 演算法裡的 hash join。

In [ ]:
# 索引不是抽象概念——它真的佔頁（U06 的 dbstat 再出動；此時只有 cust 主樹）
try:
    print(pd.read_sql_query("""SELECT name, COUNT(*) AS 頁數,
                                     ROUND(COUNT(*) * 4096 / 1e6, 1) AS 約MB
                              FROM dbstat GROUP BY name ORDER BY 頁數 DESC""", con).to_string(index=False))
    print("\n→ 等下每建一個索引，這張清單就多一棵樹、多一疊頁——空間帳單看得見。")
except Exception:
    print("（這顆 SQLite 沒編譯 dbstat；用檔案大小前後對照也能看出索引的體積）")

In [ ]:
# hash 的能與不能，用 Python dict 五秒體會
hindex = {}
for i, v in enumerate(vals):
    hindex.setdefault(v, []).append(i)             # hash index：值 → rowids

t = time.time()
for _ in range(100_000):
    hindex.get(12345)
print(f"hash 點查：{(time.time()-t)/100_000*1e9:.0f} ns／次（比 B-tree 的 µs 級又快一截）")

t = time.time()
n_range = sum(len(ids) for v, ids in hindex.items() if 12000 <= v <= 12100)   # 範圍？只能全翻 hash 桶
print(f"hash 範圍查：{(time.time()-t)*1000:.0f} ms（跟沒索引一樣——值被打散，順序死了）")
print("→ 所以磁碟世界的預設是 B-tree：點查夠快，還保住範圍、前綴、ORDER BY 三樣白吃的。")

### 隨堂練習 A（3 分鐘）

1. f=250 時，2500 萬列的樹高是多少？
2. 為什麼「葉節點串成鏈」對 `BETWEEN` 重要？拿掉它會怎樣？
3. 你的專題哪個欄位「值很長」（長字串）？當它做索引 key 對樹高有什麼影響？

<details><summary>答案</summary>

1. ⌈log₂₅₀ 2.5×10⁷⌉ = 4（250³=1.56×10⁷ 不夠、250⁴ 夠）——上一個工作區已經幫你算過。
2. 沒有鏈就要「回到上層再下來」找下一片葉——範圍掃描從順序讀退化成反覆下樹。
3. key 長 → 一頁塞的路標變少 → fanout 小 → 樹變高。長字串欄位想查「前綴」可以只索引前幾個字，或另存正規化短代碼。
</details>

# 第 2 節：把理論接上 SQLite

## 2.1 `CREATE INDEX` ＋ 讀懂 `EXPLAIN QUERY PLAN`

口訣就一條：**SCAN ＝ 全表翻、SEARCH ＝ 走樹**。看到該 SEARCH 的地方出現 SCAN，就是你出手的時候。

In [ ]:
print("── 建索引前 ──")
for r in con.execute("EXPLAIN QUERY PLAN SELECT * FROM cust WHERE cid = ?", (777_777,)):
    print("  ", r[3])
t_before = bench("SELECT * FROM cust WHERE cid = ?", (777_777,))

t = time.time()
con.execute("CREATE UNIQUE INDEX idx_cid ON cust(cid)")
con.commit()
print(f"\nCREATE INDEX 花了 {time.time()-t:.2f}s（一次性成本：掃全表＋蓋一棵樹）")

print("\n── 建索引後 ──")
for r in con.execute("EXPLAIN QUERY PLAN SELECT * FROM cust WHERE cid = ?", (777_777,)):
    print("  ", r[3])
t_after = bench("SELECT * FROM cust WHERE cid = ?", (777_777,), reps=2000)
print(f"\n{t_before:.1f} ms → {t_after*1000:.0f} µs　（快約 {t_before/t_after:,.0f} 倍；SQL 一個字沒改——宣告式的紅利）")
assert t_after < t_before

In [ ]:
# 剛剛那個 UNIQUE 有雙重身分：又是索引（快）、又是約束（唯一）——一棵樹兩份工
try:
    con.execute("INSERT INTO cust(cid, city, age, amount, d) VALUES (777777, '台中', 30, 100, '2025-01-01')")
except sqlite3.IntegrityError as e:
    print("✅ 重複 cid 被 UNIQUE INDEX 擋下 →", e)
print()
print("由此推論：PRIMARY KEY 與 UNIQUE 約束**天生自帶索引**（引擎靠那棵樹檢查唯一性）——")
print("所以「幫 PK 再建一個索引」是常見的浪費：樹已經在那了。用 PRAGMA index_list 檢查：")
for r in con.execute("PRAGMA index_list('cust')"):
    print("  ", tuple(r))

## 2.2 複合索引與「最左前綴」——最常考、最常錯

`CREATE INDEX idx ON cust(city, age)`＝先按 city 排、同 city 再按 age 排（想像電話簿：姓→名）。

| 查詢條件 | 能用 idx 嗎 | 為什麼 |
|---|---|---|
| `city = ? AND age = ?` | ✅ 全用 | 兩層都能走樹 |
| `city = ?` | ✅ 用前綴 | 「姓」排在前面 |
| `age = ?` | ❌ SCAN | 只知道「名」查電話簿——每個姓都得翻 |
| `city = ? AND age > ?` | ✅ 等值＋範圍 | 等值欄放前、範圍欄放後是設計鐵則 |

**欄位順序不是裝飾**——下一格讓計畫自己招供：

In [ ]:
con.execute("CREATE INDEX idx_city_age ON cust(city, age)")
con.commit()
for label, sql in [
    ("city= AND age=", "SELECT COUNT(*) FROM cust WHERE city='新竹' AND age=30"),
    ("只給 city=",     "SELECT COUNT(*) FROM cust WHERE city='新竹'"),
    ("只給 age=",      "SELECT COUNT(*) FROM cust WHERE age=30"),
    ("city= AND age>", "SELECT COUNT(*) FROM cust WHERE city='新竹' AND age>60"),
]:
    plan = "; ".join(r[3] for r in con.execute("EXPLAIN QUERY PLAN " + sql))
    print(f"{label:16s} → {plan}")
print()
print("→ 只給 age 的那句是 SCAN（或被迫另尋出路）——「最左前綴」沒對上，索引就當你不存在。")
print("  你的專題常用查詢長什麼樣，複合索引就照那個樣子排欄位。")

In [ ]:
# 欄序是為「你的查詢長相」服務的：反過來建 (age, city)，剛剛的殘局就翻盤
con.execute("CREATE INDEX idx_age_city ON cust(age, city)")
con.commit()
for label, sql in [
    ("只給 age=",  "SELECT COUNT(*) FROM cust WHERE age = 30"),
    ("只給 city=", "SELECT COUNT(*) FROM cust WHERE city = '新竹'"),
]:
    print(f"{label:12s} → {'; '.join(r[3] for r in con.execute('EXPLAIN QUERY PLAN ' + sql))}")
print()
print("→ 現在 age-only 走 idx_age_city、city-only 走 idx_city_age——兩張索引各養一族查詢。")
print("  但別衝動「每種順序都建一份」：2.6 的寫入帳單馬上翻倍。先看你的 App 常怎麼問。")

In [ ]:
# 兩棵樹都能走時，誰說了算？——最佳化器比價（它的價目表就是 2.5 的統計資訊）
sql = "SELECT COUNT(*) FROM cust WHERE city = '新竹' AND age = 30"
print("兩個索引都符合的查詢 →", "; ".join(r[3] for r in con.execute("EXPLAIN QUERY PLAN " + sql)))
print()
print("→ 它挑了其中一棵（通常挑「估起來要摸的列比較少」的那棵）。")
print("  想看它的帳本：ANALYZE 之後查 sqlite_stat1（2.5 節）；想指定：INDEXED BY 子句（除錯用，平常別）。")

In [ ]:
# ORDER BY 也吃索引：排序欄跟索引順序對上 → 免排序；對不上 → 多一步 TEMP B-TREE
for label, sql in [
    ("同索引順序", "SELECT * FROM cust WHERE city='新竹' ORDER BY age LIMIT 5"),
    ("另排他欄",   "SELECT * FROM cust WHERE city='新竹' ORDER BY amount LIMIT 5"),
]:
    plan = "; ".join(r[3] for r in con.execute("EXPLAIN QUERY PLAN " + sql))
    print(f"{label} → {plan}")
print("\n→ 看到 USE TEMP B-TREE FOR ORDER BY ＝ 引擎現場加班排序；排行榜頁面卡就卡在這。")

## 2.3 覆蓋索引（covering index）：連回主樹那趟都省了

次級索引查到的是 rowid，通常還要**回主樹撈整列**（一筆一次隨機 I/O）。
如果查詢要的欄位**索引裡全有**——就不用回去了，計畫會亮出 `COVERING INDEX`：

In [ ]:
sql = "SELECT age FROM cust WHERE city = '台南'"          # 只要 age，而 idx_city_age 裡就有 age！
plan = "; ".join(r[3] for r in con.execute("EXPLAIN QUERY PLAN " + sql))
print("計畫：", plan)
assert "COVERING" in plan
ms_a = bench(sql, reps=10)

sql2 = "SELECT amount FROM cust WHERE city = '台南'"       # amount 不在索引 → 每筆都要回主樹
print("計畫：", "; ".join(r[3] for r in con.execute("EXPLAIN QUERY PLAN " + sql2)))
ms_b = bench(sql2, reps=10)
print(f"\n覆蓋 {ms_a:.1f} ms vs 回表 {ms_b:.1f} ms（差 {ms_b/ms_a:.1f} 倍——十幾萬次回訪 vs 零次）")
print("→ 高頻報表查詢可以「把 SELECT 的欄位補進索引尾巴」換覆蓋——空間換時間的經典交易。")

## 2.4 運算式會殺死索引（AI 生成 SQL 的高頻事故）

索引存的是**原值**；你在欄位上套了函數，樹裡沒有那個加工後的值——只好全掃。

In [ ]:
con.execute("CREATE INDEX idx_d ON cust(d)")
con.commit()
for label, sql in [
    ("❌ 函數包住欄位", "SELECT COUNT(*) FROM cust WHERE strftime('%Y', d) = '2025'"),
    ("✅ 改寫成範圍",   "SELECT COUNT(*) FROM cust WHERE d >= '2025-01-01' AND d < '2026-01-01'"),
    ("❌ 欄位參與運算", "SELECT COUNT(*) FROM cust WHERE amount * 1.1 > 40000"),
    ("✅ 把運算移到常數那邊", "SELECT COUNT(*) FROM cust WHERE amount > 40000 / 1.1"),
]:
    plan = "; ".join(r[3] for r in con.execute("EXPLAIN QUERY PLAN " + sql))
    print(f"{label:14s} → {plan}")
print()
print("心法：**讓欄位裸著站在比較的左邊**；加工去找常數那邊做。")
print("真的非算不可？SQLite 支援「運算式索引」——把加工後的值也建成樹：")
con.execute("CREATE INDEX idx_year ON cust(strftime('%Y', d))")
con.commit()
print("  建了 idx_year 之後 →", "; ".join(
    r[3] for r in con.execute("EXPLAIN QUERY PLAN SELECT COUNT(*) FROM cust WHERE strftime('%Y', d) = '2025'")))

In [ ]:
# LIKE 前綴能吃索引嗎？——能，但有一個著名的開關
sql = "SELECT COUNT(*) FROM cust WHERE d LIKE '2025-06%'"
print("預設　　　　　　　　　 →", "; ".join(r[3] for r in con.execute("EXPLAIN QUERY PLAN " + sql)))
con.execute("PRAGMA case_sensitive_like = ON")
print("case_sensitive_like=ON →", "; ".join(r[3] for r in con.execute("EXPLAIN QUERY PLAN " + sql)))
con.execute("PRAGMA case_sensitive_like = OFF")
print()
print("→ 預設的 LIKE 不分大小寫，最佳化器不敢把 '2025-06%' 轉成範圍（怕漏掉大小寫變體）→ SCAN；")
print("  開了開關（或改用 GLOB／手寫 d >= '2025-06' AND d < '2025-07'）就 SEARCH。")
print("  搜尋框「打前綴即查」快不快，分水嶺就在這一格。")

In [ ]:
# 部分索引（partial index）：只幫「常查的那一小撮」建樹——體積小、更新省
con.execute("CREATE INDEX idx_vip ON cust(amount) WHERE city = '新竹'")
con.commit()
sql = "SELECT COUNT(*) FROM cust WHERE city = '新竹' AND amount > 20000"
print("計畫 →", "; ".join(r[3] for r in con.execute("EXPLAIN QUERY PLAN " + sql)))
try:
    n = pd.read_sql_query(
        "SELECT name, COUNT(*) AS 頁 FROM dbstat WHERE name IN ('idx_vip','idx_cid') GROUP BY name", con)
    print(n.to_string(index=False))
    print("→ 只索引 10% 的列，樹就小一號（「等待中預約唯一」這類設計也是同一招）")
except Exception:
    print("（無 dbstat：概念不變——部分索引只收錄符合 WHERE 的列）")

## 2.5 selectivity：索引不是越多越靈

**選擇性 ＝ 條件濾掉多少**。`cid = ?`（百萬分之一）超挑 → 索引神；`city = '台中'`（三成的列）不挑——
就算走索引也要回主樹三十萬次，引擎常寧可全掃（循序讀還比較快，U06 的道理）。

In [ ]:
print("各欄的「不同值數」（NDV）——選擇性的第一個線索：")
for col in ("cid", "city", "age"):
    ndv = con.execute(f"SELECT COUNT(DISTINCT {col}) FROM cust").fetchone()[0]
    print(f"  {col:6s} NDV = {ndv:>9,} → 平均每個值 {1_000_000 // ndv:>7,} 列")

t_low = bench("SELECT SUM(amount) FROM cust WHERE city = '台中'", reps=3)      # 低選擇性（30% 的列）
t_high = bench("SELECT SUM(amount) FROM cust WHERE city='台中' AND age=25", reps=10)
print(f"\n只濾 city（30 萬列中選）：{t_low:7.1f} ms ——索引在，但幫助有限")
print(f"city＋age（≈6 千列中選）：{t_high:7.1f} ms ——條件夠挑，索引才有戲")
print("\n→ 經驗法則：常用條件的 NDV 高（或組合起來高）才值得建；性別、布林欄自己單獨建索引 ≈ 白費。")
print("  引擎怎麼「知道」挑不挑？靠統計資訊（ANALYZE）——U08 拆給你看它怎麼用、什麼時候會猜錯。")

In [ ]:
# 引擎憑什麼「估」選擇性？——ANALYZE 蒐集的統計資訊，就存在一張普通的表裡
con.execute("ANALYZE")
print(pd.read_sql_query("SELECT * FROM sqlite_stat1 ORDER BY tbl, idx", con).to_string(index=False))
print()
print("→ stat 欄讀法：「總列數 每個第1欄值平均幾列 每個(第1,2欄)組合平均幾列…」")
print("  最佳化器就拿這幾個數字猜「走哪條路便宜」——U08 現場看它猜對與猜錯。")

## 2.6 索引的帳單：寫入變慢、檔案變大

In [ ]:
def insert_bench(n_idx):
    wcon = sqlite3.connect(":memory:")
    wcon.execute("CREATE TABLE t(a INTEGER, b INTEGER, c TEXT)")
    for i in range(n_idx):
        wcon.execute(f"CREATE INDEX ix{i} ON t({'a' if i==0 else 'b' if i==1 else 'c'})")
    rows = [(i, i * 7 % 1000, f"row{i}") for i in range(100_000)]
    t = time.time()
    with wcon:
        wcon.executemany("INSERT INTO t VALUES (?,?,?)", rows)
    wcon.close()
    return time.time() - t

for k in (0, 1, 3):
    print(f"{k} 個索引：插入 10 萬列 {insert_bench(k)*1000:6.0f} ms")
print()
print("→ 每個索引 ＝ 每筆寫入多維護一棵樹。讀寫比懸殊的日誌表／流水表，索引要克制。")
size_mb = os.path.getsize("million.db") / 1e6
print(f"另外 million.db 現在 {size_mb:.0f} MB——幾個索引把檔案養胖了一圈（空間也是帳單的一部分）。")

## 2.6b FK 欄位：SQLite **不會**自動建索引的大地雷

`REFERENCES` 只建約束、不建索引——而 join 與級聯檢查天天走 FK 欄。實測差多少：

In [ ]:
# FK 欄沒索引：join 每列都全掃右表；建了索引：走樹。同一句 join，天壤之別
fcon = sqlite3.connect(":memory:")
fcon.executescript("""
CREATE TABLE dept(did INTEGER PRIMARY KEY, dname TEXT);
CREATE TABLE emp(eid INTEGER PRIMARY KEY, did INTEGER REFERENCES dept(did), pay INTEGER);
""")
rng2 = np.random.default_rng(3)
fcon.executemany("INSERT INTO dept VALUES (?,?)", [(i, f"部門{i}") for i in range(200)])
with fcon:
    fcon.executemany("INSERT INTO emp VALUES (?,?,?)",
                     [(i, int(rng2.integers(0, 200)), int(rng2.integers(30, 90))) for i in range(200_000)])

sql = """SELECT d.dname, COUNT(*) FROM dept d JOIN emp e ON e.did = d.did
         WHERE d.did < 5 GROUP BY d.did"""
def timeit_once(c, q):
    t = time.time(); c.execute(q).fetchall(); return (time.time() - t) * 1000

t_nofk = timeit_once(fcon, sql)
print("FK 無索引：", "; ".join(r[3] for r in fcon.execute("EXPLAIN QUERY PLAN " + sql)))
fcon.execute("CREATE INDEX idx_emp_did ON emp(did)")
t_fk = timeit_once(fcon, sql)
print("FK 有索引：", "; ".join(r[3] for r in fcon.execute("EXPLAIN QUERY PLAN " + sql)))
print(f"\n{t_nofk:.1f} ms → {t_fk:.1f} ms（快 {t_nofk/max(t_fk,0.01):.0f} 倍）")
fcon.close()
print("→ 建表 SOP：每條 REFERENCES 寫完，順手 CREATE INDEX ON 子表(fk欄)。你的專題現在就去補。")
print("  （U03 那些 join 為什麼還算快？SQLite 臨時幫你蓋 AUTOMATIC COVERING INDEX——一次性的救火，不是常態解。）")

## 2.7 什麼時候建索引：上場前的 checklist

- [ ] 這條查詢**夠頻繁**（每頁都跑）或**夠痛**（>100 ms）才出手——先量再建（下一節的方法）。
- [ ] `WHERE`／`JOIN ON`／`ORDER BY` 用到的欄位組合，照「等值前、範圍後」排複合索引。
- [ ] 條件（組合後）選擇性夠高；布林、性別欄別單獨建。
- [ ] 高頻小查詢考慮**覆蓋**；條件固定的子集考慮**部分索引**（`CREATE INDEX … WHERE status='等待'`）。
- [ ] 別忘了帳單：寫入變慢、檔案變大；流水表克制。
- [ ] **FK 欄位自己建**（2.6b 剛量過）；PK／UNIQUE 已自帶，別重複建。
- [ ] 建完看 `EXPLAIN QUERY PLAN` 確認真的被用上（最左前綴、運算式陷阱）。

## ★ 2.8 教師示範：把「索引快多少」做成統計實驗

工程師說「有索引比較快」；統計系要說「**快多少、隨資料量怎麼變、何時開始值得**」。
實驗設計（你的專題共同要求 6 是它的一格縮影）：

| 要素 | 設定 |
|---|---|
| 因子 | 資料量 N ∈ {10³, 10⁴, 10⁵, 10⁶}（log 等距）× 索引 {無, 有} |
| 反應變數 | 點查耗時（同一批查詢的**中位數**——長尾用中位數，U05 的老朋友） |
| 控制 | 同一份母資料抽前 N 列、同一組查詢鍵、預熱一次再量 |
| 重複 | 每格重複多次取中位，壓隨機波動 |

In [ ]:
#@title 🧪 跑實驗（約 10 秒）：4 個 N × 有/無索引 × 重複量測
import statistics

rows_out = []
for N in (1_000, 10_000, 100_000, 1_000_000):
    con.execute("DROP TABLE IF EXISTS t_exp")
    con.execute(f"CREATE TABLE t_exp AS SELECT * FROM cust LIMIT {N}")
    con.commit()
    keys = [int(k) for k in np.random.default_rng(1).integers(1, N + 1, 60)]

    def median_ms(reps_each):
        samples = []
        for k in keys:
            t = time.time()
            for _ in range(reps_each):
                con.execute("SELECT * FROM t_exp WHERE cid = ?", (k,)).fetchall()
            samples.append((time.time() - t) / reps_each)
        return statistics.median(samples) * 1000          # ms

    con.execute("SELECT * FROM t_exp WHERE cid = 1").fetchall()      # 預熱
    t_noidx = median_ms(reps_each=max(1, 200_000 // N))
    con.execute("CREATE INDEX ix_exp ON t_exp(cid)"); con.commit()
    t_idx = median_ms(reps_each=50)
    rows_out.append((N, t_noidx, t_idx))
    print(f"N={N:>9,}   無索引 {t_noidx:9.3f} ms   有索引 {t_idx:.3f} ms   （差 {t_noidx/t_idx:,.0f} 倍）")

results = pd.DataFrame(rows_out, columns=["N", "scan_ms", "index_ms"])

In [ ]:
# log-log 圖＋斜率估計：讓「O(N) vs O(log N)」自己現形
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4))
ax.loglog(results.N, results.scan_ms, "o-", label="full scan")
ax.loglog(results.N, results.index_ms, "s-", label="B-tree index")
ax.set_xlabel("rows (N)"); ax.set_ylabel("median lookup (ms)")
ax.set_title("Point query: scan vs index (log-log)")
ax.grid(True, which="both", alpha=.3); ax.legend()
plt.tight_layout(); plt.show()

slope = np.polyfit(np.log10(results.N), np.log10(results.scan_ms), 1)[0]
print(f"全掃線在 log-log 上的斜率 ≈ {slope:.2f} → 快接近 1，即 O(N)：資料十倍、時間十倍")
print("索引線幾乎躺平 → O(log N)：資料千倍，時間只多一點點")
print()
print("交叉點分析：N 很小時兩線幾乎黏著（全掃千列本來就快、索引還多一層樹）；")
print("約在 10⁴–10⁵ 之後鴻溝拉開——所以專題要求你合成一萬列以上，效能故事才演得出來。")
assert slope > 0.6 and (results.scan_ms / results.index_ms).iloc[-1] > 50

In [ ]:
# 進階版：每格重複 5 輪，畫中位數＋IQR 誤差條——「量測也有分佈」才是完整的實驗報告
n_rounds = 5
err_data = {"N": [], "med": [], "q1": [], "q3": []}
for N in (1_000, 10_000, 100_000):
    con.execute("DROP TABLE IF EXISTS t_e2")
    con.execute(f"CREATE TABLE t_e2 AS SELECT * FROM cust LIMIT {N}")
    samples = []
    for rnd in range(n_rounds):
        keys = [int(k) for k in np.random.default_rng(rnd).integers(1, N + 1, 30)]
        t = time.time()
        for k in keys:
            con.execute("SELECT * FROM t_e2 WHERE cid = ?", (k,)).fetchall()
        samples.append((time.time() - t) / 30 * 1000)
    samples.sort()
    err_data["N"].append(N)
    err_data["med"].append(samples[len(samples) // 2])
    err_data["q1"].append(samples[1]); err_data["q3"].append(samples[-2])

e = pd.DataFrame(err_data)
fig, ax = plt.subplots(figsize=(6, 3.2))
ax.errorbar(e.N, e.med, yerr=[e.med - e.q1, e.q3 - e.med], fmt="o-", capsize=4)
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("rows (N)"); ax.set_ylabel("scan ms (median, IQR bars)")
ax.set_title("Measurement has a distribution too")
plt.tight_layout(); plt.show()
print(e.round(3).to_string(index=False))
print("→ 報一個點估計不如報「中位數＋散佈」——把統計素養帶進效能報告，是你們的主場優勢。")

### 這一段給統計系的弦外之音

- 量測要**重複**、比較要**同控制條件**、長尾分佈報**中位數**、尺度跨級就上 **log-log**——
  這不是資料庫技巧，是你們的本行實驗設計；拿去用在任何「A 比 B 快」的爭論上。
- 想更講究：每格多做幾輪算 IQR 畫誤差條、換不同 key 分佈（熱門 vs 均勻）當第三個因子——
  這就是一篇小小的實驗報告了。

### 索引迷思三連發（口試常見）

1. **「每個欄位都建一個就對了」**——寫入帳單 ×N、優化器還可能選錯；照 2.7 checklist 建「查詢長相」需要的。
2. **「有索引＝一定變快」**——低選擇性（2.5）、函數包欄位（2.4）、最左前綴沒對上（2.2）三種情況它都在旁邊發呆。
3. **「索引會自己保持最新，很划算」**——「自己保持最新」正是代價（每筆寫入同步維護 B-tree）；讀多寫少才划算。

### 隨堂練習 B（3 分鐘，紙上）：幫 sales.db 的三句報表開藥單

U03 的電商庫 `orders(oid, cid, pid, odate, qty, amount)`，高頻查詢：
(a) `WHERE cid = ? ORDER BY odate DESC LIMIT 10`（某顧客最近訂單）
(b) `WHERE odate BETWEEN ? AND ?`（期間報表）
(c) `WHERE pid = ? AND odate >= ?`（某商品近況）

<details><summary>參考藥單</summary>

(a) `(cid, odate)`——等值前、排序欄跟上，連 ORDER BY 都免加班；
(b) `(odate)`；
(c) `(pid, odate)`——等值前、範圍後。
每張都要能講出「服務哪句查詢」——說不出的索引就別建。
</details>

In [ ]:
# 練習 B 驗證工作區：不用等下課——把 sales.db 之外的替身現場做出來驗藥單
vcon = sqlite3.connect(":memory:")
vcon.execute("CREATE TABLE orders(oid INTEGER PRIMARY KEY, cid INT, pid INT, odate TEXT, amount INT)")
r3 = np.random.default_rng(9)
with vcon:
    vcon.executemany("INSERT INTO orders VALUES (?,?,?,?,?)",
        [(i, int(r3.integers(1, 3000)), int(r3.integers(1, 200)),
          str(np.datetime64('2025-01-01') + int(r3.integers(0, 365))), int(r3.integers(50, 3000)))
         for i in range(100_000)])
# TODO：①先 EXPLAIN 三句查詢（都是 SCAN）②照藥單 CREATE INDEX ③再 EXPLAIN 對答案
sql_a = "SELECT * FROM orders WHERE cid = 7 ORDER BY odate DESC LIMIT 10"
print("建索引前：", "; ".join(r[3] for r in vcon.execute("EXPLAIN QUERY PLAN " + sql_a)))
# vcon.execute("CREATE INDEX ...")




# 課堂實作（35 分鐘）

## A. 索引偵探（15 分）：先寫下預測，再跑 EXPLAIN 對答案

用上面的 `cust`（已有 idx_cid、idx_city_age、idx_age_city、idx_d、idx_year、idx_vip）。每題先猜：SCAN？SEARCH？COVERING？TEMP B-TREE？

In [ ]:
quiz_sqls = [
    "SELECT * FROM cust WHERE age = 40",
    "SELECT * FROM cust WHERE city = '桃園' AND age BETWEEN 30 AND 35",
    "SELECT age FROM cust WHERE city = '桃園'",
    "SELECT * FROM cust WHERE d BETWEEN '2025-06-01' AND '2025-06-30'",
    "SELECT * FROM cust WHERE upper(city) = '台中'",
    "SELECT * FROM cust WHERE city = '高雄' ORDER BY amount DESC LIMIT 10",
]
for i, sql in enumerate(quiz_sqls, 1):
    print(f"Q{i}: {sql}")
    # 先在心裡（或紙上）寫預測，再取消下一行註解對答案：
    # print("    →", "; ".join(r[3] for r in con.execute("EXPLAIN QUERY PLAN " + sql)))
print("\n（解答格在下面——先猜完再開！）")

<details><summary>📖 偵探解答與講評</summary>

1. **SEARCH idx_age_city**——本來 `age` 只當 idx_city_age 的第二欄會 SCAN；但 2.2 我們補建了 (age, city)，於是有樹可走。想看 SCAN 版：`DROP INDEX idx_age_city` 再跑一次。
2. **SEARCH idx_city_age**——等值＋範圍，教科書用法。
3. **COVERING INDEX**——要的 `age` 就在索引裡，免回主樹。
4. **SEARCH idx_d**——日期存 ISO 字串的紅利：範圍即字串範圍。
5. **SCAN**——`upper()` 包住欄位，樹裡沒有加工值（除非建運算式索引）。
6. **SEARCH idx_city_age ＋ USE TEMP B-TREE FOR ORDER BY**——找得快，但 `amount` 排序要現場加班；
   若這頁是高頻排行榜，考慮 `(city, amount)` 索引——下一格當場驗證。
</details>

In [ ]:
# 偵探 Q6 的續集：把「TEMP B-TREE 加班」用一張 (city, amount) 索引救掉
sql6 = "SELECT * FROM cust WHERE city = '高雄' ORDER BY amount DESC LIMIT 10"
print("救援前 →", "; ".join(r[3] for r in con.execute("EXPLAIN QUERY PLAN " + sql6)))
before6 = bench(sql6, reps=5)
con.execute("CREATE INDEX idx_city_amount ON cust(city, amount)")
con.commit()
print("救援後 →", "; ".join(r[3] for r in con.execute("EXPLAIN QUERY PLAN " + sql6)))
after6 = bench(sql6, reps=200)
print(f"{before6:.1f} ms → {after6:.3f} ms——排序欄放進索引尾巴，TOP-N 排行榜從加班變白吃")
assert after6 < before6

## B. 對你自己的專題做（20 分）——共同要求 6 當堂完成

1. 找出你 App **最慢或最頻繁**的查詢（報表那幾句通常就是）；
2. `EXPLAIN QUERY PLAN` 留下「前」的證據 → 建索引 → 留下「後」的證據；
3. 計時對照（模仿 2.1 的 `bench()`；資料不足一萬列先把合成資料灌滿）；
4. 把「前後計畫＋前後毫秒」四行貼進你的專題 notebook——這一節就算完工。

In [ ]:
# 課堂實作工作區：你的專題效能對照
mycon = sqlite3.connect("myapp.db")

# TODO ①：你的最慢查詢
# TODO ②：EXPLAIN QUERY PLAN（前）
# TODO ③：CREATE INDEX ...
# TODO ④：EXPLAIN QUERY PLAN（後）＋ 計時對照
print("工作區就緒")

## 專題進度建議（非繳交）

對照 [syllabus](https://github.com/chang-ye-tu/db/blob/master/syllabus.md) 進度表：**到 U07——索引效能對照完成、報表查詢補齊**。

1. 課堂實作 B 的四行證據進 notebook（共同要求 6 ✅）；
2. 報表湊滿 5 張（U03 的 window／樞紐／補零招式輪一遍），每張配一句「回答什麼問題」；
3. 對照 checklist（2.7）掃一次你的 schema：**FK 欄有索引嗎**（2.6b 的地雷）？流水表是不是建太多？
4. 下個單元起沒有新的「必交技能」——U08、U09 是內幕與報告準備，把握時間組裝與排練。

# 本單元你應該帶走

1. 索引＝排序＋二分的樹版本：**B+ tree** 葉存資料、內節點放路標、爆了就分裂往上長——永遠平衡、樹高 ⌈log_f N⌉（百萬列 ≈ 3 頁）。
2. `EXPLAIN QUERY PLAN` 口訣：**SCAN 全翻、SEARCH 走樹**；`COVERING` 免回表；`TEMP B-TREE` 是現場加班排序（排行榜救法：排序欄進索引尾巴）。
3. 複合索引**最左前綴**：等值前、範圍後；`ORDER BY` 順著索引白吃排序；兩樹皆可走時最佳化器按統計比價。
4. **別讓函數包住欄位**——改寫成範圍，或建運算式索引；LIKE 前綴有 case_sensitive_like 開關。
5. 選擇性決定值不值得：NDV 低的欄別單獨建；**FK 欄 SQLite 不會自動建**（實測差百倍），自己來；PK/UNIQUE 自帶樹，別重複建。
6. 索引的帳單：寫入每筆多維護一棵樹、檔案變大——先量再建。
7. 「快多少」要用實驗說話：重複量測、中位數、log-log、交叉點——統計系的本行。

**下個單元**：查詢處理——SQL 的一生（parser→代數→計畫→執行）、**現場手造迷你 SQL 引擎**、join 三演算法對決、以及最佳化器什麼時候會猜錯。讀物：Silberschatz ch14（本單元）、ch15–16（預習）；Ullman ch14；Winand《SQL Performance Explained》。

---
## 附錄 A：本單元 cheatsheet

```sql
CREATE INDEX idx ON t(a, b);          -- 複合：等值欄前、範圍欄後；最左前綴才有效
CREATE UNIQUE INDEX ux ON t(a);       -- 唯一約束的另一張臉（PK/UNIQUE 自帶，別重複建）
CREATE INDEX px ON t(a) WHERE st='等待';      -- 部分索引：只索引常查的子集
CREATE INDEX ex ON t(strftime('%Y', d));      -- 運算式索引：救回被函數包住的查詢
CREATE INDEX fx ON child(fk_col);     -- FK 欄自己建（SQLite 不會自動）
EXPLAIN QUERY PLAN SELECT ...;        -- SCAN=全掃  SEARCH=走樹  COVERING=免回表
DROP INDEX idx;
PRAGMA index_list('t');  PRAGMA index_info('idx');
ANALYZE;                              -- 更新統計資訊（U08：最佳化器靠它猜）
```

```python
# 計時模板（專題共同要求 6 直接抄）
def bench(sql, params=(), reps=200):
    t = time.time()
    for _ in range(reps):
        con.execute(sql, params).fetchall()
    return (time.time() - t) / reps * 1000    # ms/次
```

避雷五句：函數別包欄位・最左前綴要對上・低 NDV 別單建・FK 欄自己建・PK 別重複建。

### 附錄 A2：索引管理小抄

- 命名慣例：`idx_表_欄1_欄2`（部分索引加語意後綴）；跟 schema 放同一格，重跑即重建。
- 檢查現況：`PRAGMA index_list('t')`／`SELECT * FROM sqlite_master WHERE type='index'`。
- 大改資料後：`ANALYZE`（讓統計跟上）；懷疑樹碎了才 `REINDEX`（很少用得到）。
- 專題交件前：把「為什麼建這幾個索引」寫成 2–3 行設計說明——口試必問。

## 附錄 B：讀物地圖（本單元）

| 講義小節 | Silberschatz 7e | Garcia-Molina/Ullman/Widom 2e |
|---|---|---|
| 索引概念、dense/sparse | §14.1–14.2 | §14.1 |
| B+ tree 與分裂 | §14.3 | §14.2 |
| hash | §14.5、§24.5 | §14.3 |
| 複合／覆蓋索引實務 | §14.4、§14.9 | §14.4 |
| Winand《SQL Performance Explained》 | ch1–2（最左前綴講得最好） | —— |